# ErgoVision — Ergonomic Risk Assessment Pipeline

Vision-based ergonomic risk assessment using YOLOv8-pose + RULA-inspired scoring.

**Pipeline**:  Image/Video → YOLOv8-pose → keypoints → RULA-inspired risk scoring → classification

---

## Section 1: Environment Setup

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy matplotlib

^C


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aider-chat 0.86.2 requires huggingface-hub==1.4.1, but you have huggingface-hub 0.36.2 which is incompatible.
aider-chat 0.86.2 requires numpy==1.26.4, but you have numpy 2.4.6 which is incompatible.
aider-chat 0.86.2 requires tokenizers==0.22.2, but you have tokenizers 0.21.4 which is incompatible.
captum 0.8.0 requires numpy<2.0, but you have numpy 2.4.6 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
gradio 5.47.2 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
gradio 5.47.2 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
tensorflow 2.19.1 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.6 which is incompatible.
numba 0.59.1 requires numpy<1.27,>=1.22, but yo

In [ ]:
import sys
sys.path.insert(0, '.')

import ergovision
from ergovision import (
    ErgoPipeline, PoseEstimator, ErgonomicScorer,
    Assembly101Loader, VideoFrameExtractor,
    RobustnessMetrics, ExperimentalReports,
)
print(f'ErgoVision v{ergovision.__version__} loaded')

## Section 2: Dataset Configuration

Configure paths for both the Kaggle posture dataset and Assembly101 videos.

In [ ]:
import os
from pathlib import Path

# ---- Kaggle Posture Dataset ----
kaggle_candidates = [
    Path('/kaggle/input'),
    Path('../input'),
    Path('.'),
]

kaggle_dataset_path = None
for p in kaggle_candidates:
    if p.exists():
        for root, dirs, files in os.walk(p):
            if any(f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')) for f in files):
                kaggle_dataset_path = Path(root)
                break
        if kaggle_dataset_path:
            break

if kaggle_dataset_path:
    print(f'Kaggle dataset found at: {kaggle_dataset_path}')
else:
    print('Kaggle dataset not found (expected for Assembly101-only runs).')

# ---- Assembly101 Video Dataset ----
# Set this to the path where Assembly101 videos are stored.
# On Kaggle this would be a dataset mount point.
# Change this path to match your local setup.
ASSEMBLY101_VIDEO_PATH = Path('/kaggle/input/assembly101')  # update as needed

if ASSEMBLY101_VIDEO_PATH.exists():
    print(f'Assembly101 path found at: {ASSEMBLY101_VIDEO_PATH}')
else:
    print(f'Assembly101 path not found at {ASSEMBLY101_VIDEO_PATH}.')
    print('  To run Assembly101 experiments, set ASSEMBLY101_VIDEO_PATH correctly.')

## Section 3: Baseline Posture Dataset Experiment (Experiment 1)

Run on a configurable subset of the Kaggle posture image dataset.

In [ ]:
if kaggle_dataset_path:
    from ergovision.experiments.experiment_01_posture_dataset import run_experiment as run_exp1
    
    results_exp1 = run_exp1(
        dataset_path=kaggle_dataset_path,
        subset_size=50,
        verbose=True,
    )
    print(f'\nExperiment 1 complete: {len(results_exp1)} images processed.')
else:
    print('Skipping Experiment 1 — Kaggle dataset not available.')

## Section 4: Assembly101 Frame Extraction

Extract frames from Assembly101 videos with configurable sampling and quality filtering.

In [ ]:
if ASSEMBLY101_VIDEO_PATH.exists():
    from ergovision.data.assembly101_loader import Assembly101Loader
    from ergovision.data.video_frame_extractor import VideoFrameExtractor
    from ergovision.config import (
        ASSEMBLY101_DEFAULT_SAMPLING_RATE,
        ASSEMBLY101_DEFAULT_MAX_VIDEOS,
        ASSEMBLY101_DEFAULT_MAX_FRAMES_PER_VIDEO,
    )
    
    # Discover videos
    loader = Assembly101Loader(
        video_folder=ASSEMBLY101_VIDEO_PATH,
        max_videos=ASSEMBLY101_DEFAULT_MAX_VIDEOS,
    )
    videos = loader.discover_videos()
    print(f'Found {len(videos)} Assembly101 videos')
    
    # Extract frames
    extractor = VideoFrameExtractor(
        sampling_rate=ASSEMBLY101_DEFAULT_SAMPLING_RATE,  # 1 fps
        max_frames_per_video=ASSEMBLY101_DEFAULT_MAX_FRAMES_PER_VIDEO,  # 100
    )
    
    total_frames = 0
    for vid in videos[:3]:  # process first 3 for quick test
        if vid.is_valid:
            result = extractor.extract(vid.path, video_name=vid.video_name)
            total_frames += result.frames_extracted
            print(f'  {vid.video_name}: {result.frames_extracted} frames '
                  f'({result.frames_skipped_blurry} blurry skipped)')
    
    print(f'\nTotal frames extracted: {total_frames}')
else:
    print('Skipping Section 4 — Assembly101 path not configured.')

## Section 5: Assembly101 Ergonomic Inference (Experiment 2)

Full robustness experiment: frame extraction → pose → scoring → metrics.

In [ ]:
if ASSEMBLY101_VIDEO_PATH.exists():
    from ergovision.experiments.experiment_02_assembly101_robustness import run_experiment as run_exp2
    
    metrics = run_exp2(
        video_folder=ASSEMBLY101_VIDEO_PATH,
        max_videos=3,
        sampling_rate=1.0,  # 1 frame per second
        max_frames_per_video=50,  # limit for quick run
        verbose=True,
    )
    print('\nExperiment 2 complete.')
else:
    print('Skipping Experiment 2 — Assembly101 path not configured.')

## Section 6: Robustness Metrics

Display the aggregated metrics from the Assembly101 experiment.

In [ ]:
if ASSEMBLY101_VIDEO_PATH.exists():
    import csv
    from collections import Counter
    import matplotlib.pyplot as plt
    
    # Load and display robustness report
    metrics_dir = Path('outputs') / 'assembly101' / 'metrics'
    
    # Risk distribution
    risk_csv = metrics_dir / 'assembly101_risk_distribution.csv'
    if risk_csv.exists():
        with open(risk_csv) as f:
            reader = csv.reader(f)
            next(reader)  # header
            risk_data = list(reader)
        print('Risk Distribution:')
        for cls, cnt in risk_data:
            print(f'  {cls:15s}: {cnt}')
    
    # Runtime report snippet
    runtime_txt = metrics_dir / 'assembly101_runtime_report.txt'
    if runtime_txt.exists():
        content = runtime_txt.read_text(encoding='utf-8')
        # Print key lines
        for line in content.split('\n'):
            if 'pose detection rate' in line.lower() or 'inference time' in line.lower() or 'fps' in line.lower() or 'missing keypoint' in line.lower():
                print(line)
    
    # Show figures
    figures_dir = Path('outputs') / 'assembly101' / 'figures'
    for fig_name in ['assembly101_risk_distribution.png', 'assembly101_feature_availability.png']:
        fig_path = figures_dir / fig_name
        if fig_path.exists():
            from IPython.display import display, Image
            display(Image(filename=str(fig_path)))
else:
    print('Skipping Section 6 — no metrics available.')

## Section 7: Qualitative Explainability Examples (Experiment 3)

Generate annotated sample images for each risk class and failure mode.

In [ ]:
if ASSEMBLY101_VIDEO_PATH.exists():
    from ergovision.experiments.experiment_03_qualitative_explainability import run_experiment as run_exp3
    
    run_exp3(verbose=True)
    
    # Display samples
    from IPython.display import display, Image
    figures_dir = Path('outputs') / 'assembly101' / 'figures'
    for sample in ['sample_low_risk.png', 'sample_medium_risk.png', 'sample_high_risk.png']:
        path = figures_dir / sample
        if path.exists():
            display(Image(filename=str(path)))
else:
    print('Skipping Experiment 3 — no predictions available.')

## Section 8: Paper-Ready Result Summary

Display the experimental summary text.

In [ ]:
summary_path = Path('outputs') / 'assembly101' / 'metrics' / 'paper_experimental_summary.txt'
if summary_path.exists():
    print(summary_path.read_text(encoding='utf-8'))
else:
    print('Paper summary not yet generated. Run Experiment 2 first.')

---

All outputs saved to the `outputs/` directory:
- `outputs/assembly101/extracted_frames/` — extracted video frames
- `outputs/assembly101/predictions/` — per-frame predictions CSV
- `outputs/assembly101/figures/` — risk distribution, feature availability, sample images
- `outputs/assembly101/metrics/` — runtime report, robustness CSV, paper summary
- `outputs/assembly101/failure_cases/` — failure mode examples